In [6]:
from __future__ import annotations

import argparse
import json
import os
import sys
import time
import random
from typing import Annotated, Any, Callable, Dict, List, Mapping, Optional, Sequence, TypedDict

from state import ToolOptimizerState

class LLMDescriptionSummary:
    @staticmethod
    def _gen_prompt(state: ToolOptimizerState, prompt: str, query_list: list):
        best_record = state.get("best_record", None)
        if not best_record:
            return ""
        description = best_record.description
        sampled_queries = query_list
        if len(query_list) > 22:
            sampled_queries = random.sample(query_list, 20)
        length = str(min(len(description), 350))
        return prompt.replace("{{tool_description}}", description).replace("{{number}}", length).replace("{{querys}}", "、".join(sampled_queries))

    @staticmethod
    def _vertify_result(response: dict):
        
        # check response result
        if len(set(response.keys()) - set(['think', 'optimizer_description'])) > 0:
            return {}, False , "key error"
        # check optimized_description 字符串
        optimized_description = response["optimizer_description"]
        if len(optimized_description) < 100:
            return {}, False, "optimizer_description error"
        return {
            'optimizer_description': optimized_description
        }, True, ""

            


In [7]:
import yaml
from json_repair import repair_json

from utils import *
from state import *
from openai import OpenAI
from llm_client import LLMClient
from typing import Any, Callable, Dict, List, Mapping, Optional, Sequence

# 相关配置
from flow_config import FlowConfig
from prompt_registry import PromptRegistry

# 相关执行class
from nn_recall_passk import recall_passk_function

try:
    from langgraph.checkpoint.memory import InMemorySaver
    from langgraph.graph import END, START, StateGraph
    from langgraph.graph.message import add_messages
except ImportError as exc:  # pragma: no cover
    raise RuntimeError(
        "Missing langgraph dependencies. Please run: "
        "pip install langgraph langchain-core"
    ) from exc

from llm_summary_optimizer import LLMDescriptionSummary


/root/paddlejob/workspace/env_run/output/zacharychu/miniconda3/envs/server/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:17<00:00, 22.20it/s]


In [9]:
from typing import Any, Callable, Dict, List, Mapping, Optional, Sequence, Literal
class ToolOptimizerGraph:
    def __init__(
        self,
        resource_id: str,
        llm_client: Optional[LLMClient] = None,
        prompt_path: str = "../config/prompts.json",
        flow_config_path: str = "../config/agent_config.yaml",
        tools_description_path: str = "../data/summary/tool_descriptions.json",
        test_data_path: str = "../data/summary/query.json",
        checkpointer: Optional[Any] = None,
    ) -> None:
        # 设置通用的 参数
        self.prompts = PromptRegistry(prompt_path).get_promts()
        self.flow_config = FlowConfig(flow_config_path)
        self.checkpointer = checkpointer or InMemorySaver()
        self.last_tools_description_path = tools_description_path
        # 设置 tool resource_id
        self.resource_id = resource_id

        # 加载tools 这个列表
        self.tools_dict = load_tools_from_json(self.last_tools_description_path)

        # 加载测试集
        # test_data_path: str = "../data/summary/query.json"
        # load_tools_from_json(test_data_path)#
        self.query_good_all_dict = load_tools_from_json(test_data_path)
        self.query_good_dict = self.query_good_all_dict[resource_id]

        # node -> model
        self.node_model_map: Dict[str, str] = {} 
        if self.flow_config.node_model_map:
            self.node_model_map.update({str(k): str(v) for k, v in self.flow_config.node_model_map.items()})

        # node -> temperature
        self.node_temperature_map: Dict[str, float] =  {} 
        if self.flow_config.node_temperature_map:
            self.node_temperature_map.update({str(k): float(v) for k, v in self.flow_config.node_temperature_map.items()})
 
        llm_dict = {}
        if len(self.flow_config.clients) != 0:
            for k, v in self.flow_config.clients.items():
                if "tianchi" in v["base_url"]:
                    v['host'] = "tianchi-proxy.baidu-int.com"
                    v['appid'] = 'app-CdjpA4YQ'
                    llm_dict[k] = LLMClient(**v)

        # llm_client
        if llm_client:
            self.llm_client = llm_client
        else:
            self.llm_client = LLMClient(**self.flow_config.config["default_client"])

        # node_llm_client_map
        self.node_llm_clients: Dict[str, LLMClient] = {}
        if self.flow_config.node_llm_client_map:
            self.node_llm_clients.update({
                str(k): llm_dict[v] for k, v in self.flow_config.node_llm_client_map.items()
            })

    # ---------- generate text ----------

    # node -> client
    def _client_for(self, node_name: str) -> LLMClient:
        return self.node_llm_clients.get(node_name, self.llm_client)

    # node -> model name
    def _model_for(self, node_name: str) -> str:
        return self.node_model_map.get(node_name, "deepseek-v3.2")
    # node -> temperature 
    def _temperature_for(self, node_name: str) -> float:
        return self.node_temperature_map.get(node_name, 0.8)

    def _generate_text(
        self,
        node_name: str,
        prompt: str,
        max_tokens: int = 2048,
        extra_body: Dict[str, Any] = {}
    ) -> Any:
        client = self._client_for(node_name)
        response =  client.generate_text(
            prompt = prompt,
            model=self._model_for(node_name),
            temperature=self._temperature_for(node_name),
            extra_body = extra_body
        )
        return client.parse_chat_content(response)
    
    # --------- 统计和check工具 ----------------------
    def statistic_check_tool(self, 
                    state: ToolOptimizerState):
        """示例主函数：你可以替换为自己的 JSON 文件路径。"""
        best_record = state.get("best_record", None)
        if best_record is not None:
            self.last_tools_description_path = best_record.tool_path

        self.tools_dict = load_tools_from_json(self.last_tools_description_path) 
        statistic_check_version = "version_" + str(state.get("current_version_id", 0))
        print("当前执行的版本：", statistic_check_version)
        statistic_output = "../recall_outputs/summary/" + statistic_check_version + "/" + self.resource_id
    
        detaildf = recall_passk_function(self.tools_dict, self.query_good_dict, self.resource_id, statistic_output)

        # 计算 top 1 的 precision
        detaildf_top1 = detaildf[(detaildf['k']==1) & (detaildf['view']=="merged")]
        relevants = detaildf_top1['gold_ids'].to_list()
        retrieveds = detaildf_top1['recall_ids'].to_list()
        precision1 = precision_at_k_batch(retrieveds, relevants, 1)
        recall1 = recall_at_k_batch(retrieveds, relevants, 1)

        # 计算 top 3 的 precision
        detaildf_top3 = detaildf[(detaildf['k']==3) & (detaildf['view']=="merged")]
        relevants = detaildf_top3['gold_ids'].to_list()
        retrieveds = detaildf_top3['recall_ids'].to_list()
        precision3 = precision_at_k_batch(retrieveds, relevants, 3)
        recall3 = recall_at_k_batch(retrieveds, relevants, 3)

        # 统计 top 3 的结果
        tools_query = {}
        tools_case_ids = []
        for indx, row in detaildf_top3.iterrows():
            recall_ids = row["recall_ids"]
            if len(recall_ids) == 0:
                recall_ids = [NO_CALL] 
            if len(tools_query.get(recall_ids[0], [])) == 0:
                tools_query[recall_ids[0]] =  []
            tools_query[recall_ids[0]].append(row["query"])
            tools_case_ids.append(recall_ids[0])
        top_tools_case = {k: tools_query[k] for k in get_k_tool(tools_case_ids, 3)}
        tools_case_all = {k: tools_query[k] for k in set(tools_case_ids)}

        version_id, next_version_id = next_version_pair(state)

        versionInfo = VersionRecord(
            version_id = statistic_check_version, 
            parent_version_id = version_id,
            stage = "statistic",
            description = self.tools_dict[self.resource_id]["description"],
            case_result = tools_case_all,
            top_case = top_tools_case,
            tool_path = statistic_output + "/tool_prompt.json",
            recall1 = recall1,
            precision1 = precision1,
            recall3 = recall3,
            precision3 = precision3
        )

        # 判断是否需要更新最佳记录
        # 条件1: best_record 不存在 (即为 None)
        # 条件2: 新的指标 (recall3, precision3) 优于旧的 best_record
        should_update = (best_record is None) or \
                (recall3 > best_record.recall3 and precision3 > best_record.precision3)

        with open(statistic_output + "/tool_prompt.json","w") as w:
            json.dump(self.tools_dict, w, ensure_ascii=False, indent=2)

        if should_update:
            return {
                    "current_version_id": version_id,
                    "next_version_id": next_version_id,
                    "version_history": append_version_history(state, versionInfo),
                    "best_description": state.get("current_description", ""),
                    "best_version_id": state.get("current_version_id", 0),
                    "best_record": versionInfo,

                }
        else:
            return {
                "current_version_id": version_id,
                "next_version_id": next_version_id,
                "version_history": append_version_history(state, versionInfo)
            }


    # ---------  LLMNodeSummary node--------------
    def llm_description_summary(self, 
                                  state: ToolOptimizerState):
        prompt = LLMDescriptionSummary._gen_prompt(state, self.prompts['summary'], list(self.query_good_dict.keys()))

        for _ in range(3):
            response, stype, flag = dag._generate_text(node_name = "summary", prompt = prompt)
            response, flag, error_type = LLMDescriptionSummary._vertify_result(response)
            if flag:
                break
        if flag:
            optimizer_description = response["optimizer_description"]
            
            
            print("description: ", state.get("current_version_id", "-1"), self.tools_dict[self.resource_id]['description'])
            optimizer_record = InfoRecord(
                version_id = state.get("current_version_id", "-1"),
                stage = "summary",
                info = response
            )
            return {
                "optimizer_history": append_optimizer_history(state, optimizer_record)
            }


    # ---------- graph build ----------

def build_optimizer_graph():
    """Build and compile the LangGraph optimization workflow."""

    graph = StateGraph(ToolOptimizerState)
    graph.add_node("optimizer", optimizer)
    graph.add_node("vertify_after_optimizer", vertify_after_optimizer)
    graph.add_edge("bump_iteration", "optimizer")
    return graph.compile()


def should_continue(state: ToolOptimizerState) -> Literal["summary_optimizer", END]:
    # 达到最大轮次
    max_iteration = state.get("max_iterations", 3)
    if state.get("current_version_id", 10) > max_iteration:
        return END
    return "summary_optimizer"

#------- init ----
def list_file(folder_path):
    all_items = os.listdir(folder_path)
    # 只列出文件（不包括文件夹）
    files_only = [os.path.join(folder_path, item) for item in all_items if os.path.isfile(os.path.join(folder_path, item)) and item.endswith(".pkl")]
    print("\n只列出文件:")
    return files_only
resource_ids_all = []
for path in list_file("../recall_outputs/summary/"):
    resource_ids_all.append(path.split("/")[-1].replace(".pkl", ""))

query_good_all_dict = load_tools_from_json("../data/summary/query.json")
for resource_id, value in query_good_all_dict.items():
    if resource_id in resource_ids_all:
        print(resource_id, "skip!!!")
        continue
    print("开始执行：", resource_id)
    dag = ToolOptimizerGraph(resource_id = resource_id,
        prompt_path = "../config/prompts.json",
        flow_config_path = "../config/agent_config.yaml",
        tools_description_path = "../data/summary/tool_description.json",
        test_data_path  = "../data/summary/query.json",
        checkpointer = None)

    graph = StateGraph(ToolOptimizerState)
    graph.add_node("statistic_check_tool", dag.statistic_check_tool)
    graph.add_node("summary_optimizer", dag.llm_description_summary)
    graph.set_entry_point("statistic_check_tool")
    # graph.add_edge("statistic_check_tool", END)
    graph.add_conditional_edges(
        "statistic_check_tool",
        should_continue,
        {
            "summary_optimizer": "summary_optimizer",
            END: END
        }
    )
    graph.add_edge("summary_optimizer", "statistic_check_tool")
    compiled_graph = graph.compile()

    initial_state = {
            "resource_id": resource_id,
            "title": dag.tools_dict[resource_id]["title"],
            "original_description": dag.tools_dict[resource_id]["description"],
            # 当前工作基线指针（可手动/自动回滚）
            "current_version_id": 0,
            "current_description": dag.tools_dict[resource_id]["description"],
            # 下一个版本的id
            "next_version_id": 1,
            # 固定参数
            "max_iterations": 3,
            "iteration": 0,
            # 版本历史仓库：全量快照存储
            "version_history": [],
            #记录历史
            "optimizer_history": []
        }
    state = compiled_graph.invoke(initial_state)

    save_pickle(state, "../recall_outputs/summary/" + resource_id + ".pkl")                                                                                                                                                    


只列出文件:
5103 skip!!!
5359 skip!!!
36607 skip!!!
46671 skip!!!
50159 skip!!!
50160 skip!!!
50161 skip!!!
50162 skip!!!
50354 skip!!!
50709 skip!!!
50926 skip!!!
51088 skip!!!
51255 skip!!!
51590 skip!!!
5293 skip!!!
5351 skip!!!
5428 skip!!!
5432 skip!!!
5802 skip!!!
23 skip!!!
50143 skip!!!
62172 skip!!!
85 skip!!!
50133 skip!!!
50192 skip!!!
5671 skip!!!
60868 skip!!!
60872 skip!!!
60874 skip!!!
62106 skip!!!
62306 skip!!!
5851 skip!!!
4526 skip!!!
50574 skip!!!
50760 skip!!!
5271 skip!!!
5819 skip!!!
50393 skip!!!
5179 skip!!!
62248 skip!!!
62206 skip!!!
8470 skip!!!
51323 skip!!!
62016 skip!!!
51275 skip!!!
51474 skip!!!
5544 skip!!!
60338 skip!!!
60540 skip!!!
60977 skip!!!
61800 skip!!!
5881 skip!!!
5882 skip!!!
46679 skip!!!
47190 skip!!!
47198 skip!!!
开始执行： 47242
当前执行的版本： version_0
开始建库: description
Building embeddings for view=description, size=100
开始建库: title_description
Building embeddings for view=title_description, size=100
Total tools indexed: 100
Total eval queries: 0


KeyError: 'view'

In [10]:
from utils import *
from openai import OpenAI
from llm_client import LLMClient
# 相关配置
from flow_config import FlowConfig
from prompt_registry import PromptRegistry
from state import *
def list_file(folder_path):
    all_items = os.listdir(folder_path)
    # 只列出文件（不包括文件夹）
    files_only = [os.path.join(folder_path, item) for item in all_items if os.path.isfile(os.path.join(folder_path, item)) and item.endswith(".pkl")]
    print("\n只列出文件:")
    return files_only
    
resource_ids_all = []
for path in list_file("../recall_outputs/summary/"):
    resource_ids_all.append(path.split("/")[-1].replace(".pkl", ""))

query_good_all_dict = load_tools_from_json("../data/summary/query.json")
for resource_id, value in query_good_all_dict.items():
    if resource_id in resource_ids_all:
        print(resource_id, "skip!!!")
        continue
    print("开始执行：", resource_id)


只列出文件:
5103 skip!!!
5359 skip!!!
36607 skip!!!
46671 skip!!!
50159 skip!!!
50160 skip!!!
50161 skip!!!
50162 skip!!!
50354 skip!!!
50709 skip!!!
50926 skip!!!
51088 skip!!!
51255 skip!!!
51590 skip!!!
5293 skip!!!
5351 skip!!!
5428 skip!!!
5432 skip!!!
5802 skip!!!
23 skip!!!
50143 skip!!!
62172 skip!!!
85 skip!!!
50133 skip!!!
50192 skip!!!
5671 skip!!!
60868 skip!!!
60872 skip!!!
60874 skip!!!
62106 skip!!!
62306 skip!!!
5851 skip!!!
4526 skip!!!
50574 skip!!!
50760 skip!!!
5271 skip!!!
5819 skip!!!
50393 skip!!!
5179 skip!!!
62248 skip!!!
62206 skip!!!
8470 skip!!!
51323 skip!!!
62016 skip!!!
51275 skip!!!
51474 skip!!!
5544 skip!!!
60338 skip!!!
60540 skip!!!
60977 skip!!!
61800 skip!!!
5881 skip!!!
5882 skip!!!
46679 skip!!!
47190 skip!!!
47198 skip!!!
开始执行： 47242
开始执行： 47266
开始执行： 47272
开始执行： 5012
开始执行： 5778
开始执行： 5273
开始执行： 5917
开始执行： 60293
开始执行： 51712
开始执行： 267
开始执行： 5143
开始执行： 5144
开始执行： 39529
开始执行： 39537
开始执行： 4551
开始执行： 52364
开始执行： 51129
开始执行： 5550
开始执行： 4999
开始执行： 51044
开始

In [103]:
def pass_at_k(n: int, c: int, k: int) -> float:
    """
    计算 Pass@K (通常用于代码生成或生成式检索评估)
    Args:
        n: 总生成样本数
        c: 正确样本数
        k: 评估的截断阈值 (Top-K)
    Returns:
        Pass@K 得分 (0.0 ~ 1.0)
    """
    if n - c < k:
        return 1.0
    # 使用组合数公式计算至少命中一个正确结果的概率
    # 1 - C(n-c, k) / C(n, k)
    # 为避免阶乘溢出，使用连乘计算
    prob_no_pass = 1.0
    for i in range(k):
        prob_no_pass *= (n - c - i) / (n - i)
    return 1.0 - prob_no_pass


def precision_at_k(retrieved: List[Union[str, int]], relevant: List[Union[str, int]], k: int) -> float:
    """
    计算 Precision@K (Top-K 结果中相关文档的比例)
    Args:
        retrieved: 模型检索/推荐出的结果列表 (按相关性降序排列)
        relevant: 真实相关文档的集合
        k: 评估的截断阈值
    Returns:
        Precision@K 得分 (0.0 ~ 1.0)
    """
    if k <= 0:
        return 0.0
    # 截取前 k 个结果
    top_k_results = retrieved[:k]
    if not top_k_results:
        return 0.0
    
    # 计算前 k 个结果中有多少是相关的
    relevant_count = sum(1 for item in top_k_results if item in relevant)
    return relevant_count / len(relevant)

def precision_at_k_batch(retrieveds: List[List[Union[str, int]]], relevants: List[List[Union[str, int]]], k: int) -> float:
    """
    计算 Precision@K (Top-K 结果中相关文档的比例)
    Args:
        retrieved: 模型检索/推荐出的结果列表 (按相关性降序排列)
        relevant: 真实相关文档的集合
        k: 评估的截断阈值
    Returns:
        Precision@K 平均得分 (0.0 ~ 1.0)
    """
    
    if len(relevants) != len(retrieveds):
        return 0.0
    else:
        precision_scores = []
        for retrieved, relevant in zip(retrieveds, relevants):
            precision_scores.append(precision_at_k(retrieved, relevant, k))
    print("precision_scores", precision_scores)
    return sum(precision_scores) / len(retrieveds)



def recall_at_k(retrieved: List[Union[str, int]], relevant: List[Union[str, int]], k: int) -> float:
    """
    计算 Recall@K (所有相关文档中，被检索到且排在 Top-K 的比例)
    Args:
        retrieved: 模型检索/推荐出的结果列表 (按相关性降序排列)
        relevant: 真实相关文档的集合
        k: 评估的截断阈值
    Returns:
        Recall@K 得分 (0.0 ~ 1.0)
    """
    if not relevant:
        return 0.0
    
    # 截取前 k 个结果
    top_k_results = retrieved[:k]
    
    # 计算前 k 个结果中命中了多少个真实相关文档
    relevant_found = sum(1 for item in top_k_results if item in item in relevant)
    return relevant_found / len(relevant)

def recall_at_k_batch(retrieveds: List[List[Union[str, int]]], relevants: List[List[Union[str, int]]], k: int) -> float:
    """
    计算 Recall@K (所有相关文档中，被检索到且排在 Top-K 的比例)
    Args:
        retrieved: 模型检索/推荐出的结果列表 (按相关性降序排列)
        relevant: 真实相关文档的集合
        k: 评估的截断阈值
    Returns:
        Recall@K 平均得分 (0.0 ~ 1.0)
    """
    if len(relevants) != len(retrieveds):
        return 0.0
    else:
        recall_scores = []
        for retrieved, relevant in zip(retrieveds, relevants):
            recall_scores.append(recall_at_k(retrieved, relevant, k))
    print("recall_scores", recall_scores)
    return sum(recall_scores) / len(retrieveds)


In [111]:
import argparse
import json
import os
import sys
import time
from typing import Annotated, Any, Callable, Dict, List, Mapping, Optional, Sequence, TypedDict
# from llm_client import LLMClient
from openai import OpenAI

class LLMClient:
    """OpenAI-compatible chat-completions client."""
    def __init__(self, base_url, api_key, timeout, max_retry, host = None, appid = None):
        self.base_url = base_url
        self.api_key = api_key
        self.max_retry=3
        self.sleep_seconds=3
        default_headers = {}
        if not host:
            default_headers["Host"] = host
        if not appid:
            default_headers["appid"] = appid
        if len(default_headers) == 0:
            self.client = OpenAI(
                base_url=base_url,
                api_key=api_key,
                timeout=timeout,
                max_retries=max_retry,  # 这里用我们自己的重试逻辑，避免 SDK + 手动双重重试
                default_headers= default_headers
            )
        else:
            self.client = OpenAI(
                base_url=base_url,
                api_key=api_key,
                timeout=timeout,
                max_retries=max_retry,  # 这里用我们自己的重试逻辑，避免 SDK + 手动双重重试
    #             default_headers= default_headers
            )
 
    def fetch_response(self,
            query: str,
            doc: str,
            prompt: str = None,
            messages: Optional[List[Dict[str, str]]] = None,
            model: Optional[str] = None,
            temperature: Optional[float] = None,
            max_tokens: Optional[int] = None,
            extra_body: Optional[Dict[str, Any]] = None,
        ) -> Dict[str, Any]:

        if not messages:
            messages = [{"role": "user",
                        "content": prompt}]
            
        last_error = None
        request_params = {
            "model": model,
            "messages": messages,
            "temperature": temperature if temperature is not None else 0.8,
            "max_tokens": max_tokens if max_tokens is not None else 8196,
        }
        if extra_body:
            request_params["extra_body"] = extra_body
        
        print("request_params", request_params)
        print("-" * 100)
        print()
        self.client.chat.completions.create(**request_params)
        
        for retry_idx in range(self.max_retry):
            try:
                response = self.client.chat.completions.create(**request_params)

                # OpenAI SDK 返回的是对象，需要转成 dict，方便 json.dumps
                output_json = response.model_dump(mode="json")
                return output_json

            except Exception as e:
                last_error = e
                print(f"[Retry {retry_idx + 1}/{self.max_retry}] 请求失败: {e}")
                time.sleep(self.sleep_seconds)

        raise RuntimeError(f"请求重试 {self.max_retry} 次后仍失败: {last_error}")

#     def request_model(
#         self,
#         *,
#         prompt: Optional[str] = None,
#         messages: Optional[List[Dict[str, str]]] = None,
#         model: Optional[str] = None,
#         temperature: Optional[float] = None,
#         max_tokens: Optional[int] = None,
#         extra_body: Optional[Dict[str, Any]] = None,
#     ) -> Dict[str, Any]:
#         """Use requests to call an OpenAI-compatible chat-completions model API.

#         This is the lowest-level model access method. All LangGraph nodes can
#         share it, and you can also call it directly when debugging a model
#         service such as vLLM, SGLang, LMDeploy, or OpenAI-compatible gateways.

#         Args:
#             prompt: Simple user prompt. Used when `messages` is not provided.
#             messages: Full chat messages, e.g.
#                 [{"role": "system", "content": "..."},
#                  {"role": "user", "content": "..."}]
#             model: Override model name for this request.
#             temperature: Override sampling temperature.
#             max_tokens: Optional output token limit.
#             extra_body: Extra OpenAI-compatible fields, such as top_p, stop,
#                 repetition_penalty, response_format, etc.

#         Returns:
#             Raw JSON response from /chat/completions.
#         """
#         if not self.base_url:
#             raise RuntimeError("OPENAI_API_BASE is empty. Please set it first.")

#         url = self.base_url.rstrip("/")
#         if not url.endswith("/chat/completions"):
#             url = url + "/chat/completions"

#         headers = {"Content-Type": "application/json"}
#         if self.api_key:
#             headers["Authorization"] = f"Bearer {self.api_key}"

#         if messages is None:
#             messages = [{"role": "user", "content": prompt or ""}]

#         payload: Dict[str, Any] = {
#             "model": model or self.model,
#             "messages": messages,
#             "temperature": 0.2 if temperature is None else temperature,
#         }
#         if max_tokens is not None:
#             payload["max_tokens"] = max_tokens
#         if extra_body:
#             payload.update(extra_body)

#         last_error: Optional[Exception] = None
#         for attempt in range(1, self.max_retry + 1):
#             try:
#                 resp = requests.post(url, headers=headers, json=payload, timeout=self.timeout)
#                 resp.raise_for_status()
#                 return resp.json()
#             except Exception as exc:  # noqa: BLE001
#                 last_error = exc
#                 if attempt < self.max_retry:
#                     time.sleep(1)

#         raise RuntimeError(f"LLM request failed after {self.max_retry} retries: {last_error}")

    def parse_chat_content(self, response: Dict[str, Any]) -> str:
        """Extract assistant text from an OpenAI-compatible response."""
        try:
            return str(response["choices"][0]["message"]["content"])
        except (KeyError, IndexError, TypeError) as exc:
            raise ValueError(f"Invalid chat completion response: {response}") from exc

    def generate_text(
        self,
        query: str,
        doc: str,
        prompt: str = None,
        *,
        messages: Optional[List[Dict[str, str]]] = None,
        model: Optional[str] = None,
        temperature: Optional[float] = None,
        max_tokens: Optional[int] = None,
        extra_body: Optional[Dict[str, Any]] = None,
    ) -> str:
        """Call model and return only assistant content.

        `model` and `temperature` can be overridden per node, so a LangGraph
        graph can use lightweight models for routing and stronger models for
        planning/writing/critique. Internally this delegates to `request_model`,
        the shared requests-based access method.
        """
        response = self.fetch_response(
            query = query,
            doc = doc,
            prompt = prompt,
            model=model,
            messages = messages,
            temperature = temperature,
            max_tokens = max_tokens,
            extra_body = extra_body,
        )
        return {
                "query": query,
                "doc": doc,
                "request_data": self.parse_chat_content(response),
                }
if __name__ == "__main__":

    llmclient = LLMClient(
        base_url="http://10.215.195.160:8880/v1",
        api_key="zacharychu",
        timeout=60,
        max_retry=3,
    )
    
    response = llmclient.generate_text(
        query = "nihao",
        model = "Qwen3-8B",
        doc = "生成",
        messages=[
            {"role": "system", "content": "你是一个严谨、简洁的中文助手。"},
        ],
        temperature=0.2,
        max_tokens=1024,
        # extra_body 可以放 top_p、stop、response_format 等 OpenAI-compatible 参数。
        extra_body={"top_p": 0.9},
    )

    print("RAW RESPONSE:")
    print(json.dumps(response, ensure_ascii=False, indent=2))
    print("\nCONTENT:")

request_params {'model': 'Qwen3-8B', 'messages': [{'role': 'system', 'content': '你是一个严谨、简洁的中文助手。'}], 'temperature': 0.2, 'max_tokens': 1024, 'extra_body': {'top_p': 0.9}}
----------------------------------------------------------------------------------------------------

RAW RESPONSE:
{
  "query": "nihao",
  "doc": "生成",
  "request_data": "好的，我将以严谨、简洁的方式为您提供帮助。请直接提出您的问题或需求。"
}

CONTENT:


In [1]:
import os
import pickle
import json
import pandas as pd

In [2]:
# env_run/output/zacharychu/code/tool_description_optimizer/recall_outputs/summary
def list_file(folder_path):
    all_items = os.listdir(folder_path)
    # 只列出文件（不包括文件夹）
    files_only = [os.path.join(folder_path, item) for item in all_items if os.path.isfile(os.path.join(folder_path, item)) and item.endswith(".pkl")]
    print("\n只列出文件:")
    return files_only
    
def load_pickle(filepath):
    with open(filepath, 'rb') as f:
        return pickle.load(f)
paths = list_file("../recall_outputs/summary")
print(paths)


只列出文件:
['../recall_outputs/summary/5179.pkl', '../recall_outputs/summary/51323.pkl', '../recall_outputs/summary/50159.pkl', '../recall_outputs/summary/51088.pkl', '../recall_outputs/summary/4551.pkl', '../recall_outputs/summary/50192.pkl', '../recall_outputs/summary/60977.pkl', '../recall_outputs/summary/50709.pkl', '../recall_outputs/summary/5671.pkl', '../recall_outputs/summary/62016.pkl', '../recall_outputs/summary/50393.pkl', '../recall_outputs/summary/5819.pkl', '../recall_outputs/summary/51044.pkl', '../recall_outputs/summary/51376.pkl', '../recall_outputs/summary/62248.pkl', '../recall_outputs/summary/62306.pkl', '../recall_outputs/summary/46679.pkl', '../recall_outputs/summary/62206.pkl', '../recall_outputs/summary/267.pkl', '../recall_outputs/summary/4526.pkl', '../recall_outputs/summary/60872.pkl', '../recall_outputs/summary/5868.pkl', '../recall_outputs/summary/52523.pkl', '../recall_outputs/summary/51474.pkl', '../recall_outputs/summary/8470.pkl', '../recall_outputs/summar

In [20]:
# result_tmp = load_pickle("../recall_outputs/summary/5179.pkl")
result_tmp["optimizer_history"]

[InfoRecord(version_id=1, stage='summary', info={'optimizer_description': '该工具提供特定歌手、艺人或主题的精选歌曲集合。用户可通过输入歌手姓名、艺名或歌曲相关主题词，获取对应的歌曲列表。列表包含歌曲名称、歌手、所属专辑及封面图等信息，并直接集成播放功能。支持查看歌曲详情、一键播放全部或单曲，并可跳转至更完整的音乐合集页面。适用于快速查找并收听某位歌手的全部歌曲、经典代表作、热门合集或特定主题下的音乐，满足用户基于人物或主题进行歌曲检索、列表浏览与即时播放的核心需求。'}),
 InfoRecord(version_id=2, stage='summary', info={'optimizer_description': '该工具提供按特定歌手、音乐主题或风格聚合的歌曲集合列表，用于快速查找和播放相关歌曲。核心功能是展示结构化歌曲信息，包括歌曲名称、歌手、所属专辑及封面图。用户可通过工具直接播放单首歌曲或一键播放全部歌曲，并支持跳转查看更多同类型歌曲或进入个人音乐库。适用于需要查找某歌手经典作品、热门歌曲排行、特定风格曲目集合等场景，如查询“陈慧娴歌曲”、“霉霉最火的十首歌”、“经典老歌”等，能高效满足用户对特定范围歌曲的检索、列举与收听需求。'}),
 InfoRecord(version_id=3, stage='summary', info={'optimizer_description': '该工具用于提供特定类型的歌曲集合，满足用户根据歌手、经典程度、热门排行或歌曲类别进行检索的需求。它能返回如“肖战最火的歌”、“蔡琴经典老歌”、“杰克逊全部歌曲”等主题的歌曲列表。工具展示歌曲名称、歌手、所属专辑、封面图等核心信息，并支持直接播放单首歌曲或播放整个列表。无论是寻找某位歌手的全部作品、最热门的十首歌，还是特定风格（如山歌、对唱）的曲目合集，该工具都能快速聚合相关歌曲并提供便捷的播放入口，帮助用户高效发现和聆听目标音乐集合。'})]

In [39]:
description_dict = {}
for path in paths:
    result = load_pickle(path)
    resource_id = result['resource_id']
    original_description = result["original_description"]
    best_description = result["best_description"]
    description = ""
    if best_description != original_description:
        description = best_description
    for optmizer in result["optimizer_history"]:
        if "InfoRecord" in str(optmizer.__class__):
            description = optmizer.info.get('optimizer_description')
            break
    description_dict[resource_id] = description

In [36]:
result["optimizer_history"]

[VersionRecord(version_id='version_0', stage='statistic', description='该工具用于游戏场景返回精准游戏、泛化游戏下载资源。工具返回内容：游戏名、icon、四要素、下载按钮、视频、图片合集、游戏简介、游戏评分、游戏标签\n组件强满足样式：\n（1）视频/图片合集\n（2）游戏名称\n（3）icon\n（4）下载入口\n（5）四要素\n（6）游戏简介、游戏评分、游戏标签\n组件交互样式：\n（1）点击游戏名称、视频/图片合集、icon、四要素（公司、版本）、简介、评分、标签跳转游戏详情页\n（2）点击隐私、权限跳转对应隐私、应用权限页面\n（3）点击下载按钮，下载游戏资源同时跳转游戏详情页', tool_path='../recall_outputs/optimizer/version_0/5868/tool_prompt.json', top_case={'52418': ['4399游戏盒', 'taptap', 'taptap下载'], '33809': ['steam'], '61468': ['我的世界']}, case_result={'61468': ['我的世界'], '52418': ['4399游戏盒', 'taptap', 'taptap下载'], '51261': ['我的世界网易官方正版'], '33809': ['steam'], '5801': ['虫虫助手']}, parent_version_id=1, recall1=0.0, precision1=0.0, recall3=0.19047619047619047, precision3=0.19047619047619047, accepted=False, reason=''),
 InfoRecord(version_id=1, stage='optimizer', info={'optimizer_description': '该工具用于游戏场景，提供游戏下载资源，包括精准推荐和泛化推荐。返回内容包含游戏名称、游戏图标（icon）、四要素（如公司、版本）、下载按钮、游戏视频、游戏图片合集、游戏简介、游戏评分、游戏标签。支持用户通过游戏平台、应用商店或下载渠道（如Steam、TapTap、4399游戏盒）查询游

In [18]:
description_dict

{'5179': '该工具提供特定歌手、艺人或主题的精选歌曲集合。用户可通过输入歌手姓名、艺名或歌曲相关主题词，获取对应的歌曲列表。列表包含歌曲名称、歌手、所属专辑及封面图等信息，并直接集成播放功能。支持查看歌曲详情、一键播放全部或单曲，并可跳转至更完整的音乐合集页面。适用于快速查找并收听某位歌手的全部歌曲、经典代表作、热门合集或特定主题下的音乐，满足用户基于人物或主题进行歌曲检索、列表浏览与即时播放的核心需求。',
 '51323': '本工具提供两类传统文化与规范书写信息的查询与展示。一是二十四节气知识查询，完整列出立春至大寒的全部节气，并按春、夏、秋、冬四季进行分类组织，清晰展示每个节气的标准名称及其对应的公历日期范围，便于用户系统性了解节气时序。二是中文数字规范写法查询，提供财务常用的大写数字（如壹、贰、叁）与其对应的小写数字（一、二、三）的对照表，并包含田字格书写样式展示，辅助用户掌握数字的标准、规范书写格式。工具核心功能在于结构化、可视化地呈现节气时序和数字书写这两类实用文化信息，适用于节气知识学习、日期查询、财务票据填写参考、汉字书写规范查阅等场景。',
 '50159': '该工具提供实时汇率查询与货币换算功能。支持全球主流货币（如美元、港币、泰铢、韩元、英镑、澳元等）与人民币之间的实时汇率查询与金额换算。工具返回内容包括：实时汇率、货币兑换金额计算结果、汇率走势图表。核心功能为：查询任意两种货币间的实时汇率；输入特定金额，快速计算兑换后的等值金额；查看人民币兑其他货币的汇率历史走势图。适用于出国旅行、海外购物、跨境投资、外贸结算等需进行货币兑换或汇率查询的场景，帮助用户快速解决汇率计算、成本估算、资金规划等问题。操作便捷，数据实时更新。',
 '51088': '该工具提供黄金期货的实时行情数据，包括最新价格（元/克）和涨跌幅。适用于查询黄金期货实时价格、追踪黄金市场动态的需求。不覆盖股票、股市大盘、全球股指等证券行情查询。用户可通过工具获取当天黄金期货最新价，并跳转至专业行情网站查看详情。',
 '4551': '输入院校名称、类型（如985/211/职业院校）、地区或分数线等条件，查询符合条件的院校榜单。返回列表包含院校名称、地区、办学层次、录取分数线、专业、招生章程等关键信息，用于快速检索高校详情、比较录取要求或筛选目标学校。',
 '501